# D2Mamba Research — Notebook A: Branches 1 & 2 (`exp/1-mambairv2-gpu-scan` + `exp/2-physics-attenuation-scan`)

**Kaggle Settings**:
- Accelerator: **GPU T4 x2** (or GPU T4 / P100)
- Internet: **ON**
- Datasets attached: **UIEB** (`raw-890`/`reference-890`) + **LSUI** (`input`/`GT`)
- Protocol: **From scratch, 200 epochs, seed=42, official zero-leakage test splits** (fair apples-to-apples comparison with Branches 3 & 4 in Notebook B).

In [ ]:
REPO = "https://github.com/asserelzeki6/D2Mamba-Research.git"

!rm -rf /kaggle/working/exp1 /kaggle/working/exp2
!git clone --depth 1 -b exp/1-mambairv2-gpu-scan {REPO} /kaggle/working/exp1
!git clone --depth 1 -b exp/2-physics-attenuation-scan {REPO} /kaggle/working/exp2

!cd /kaggle/working/exp1 && python3 verify_branch.py --device cuda
!cd /kaggle/working/exp2 && python3 verify_branch.py --device cuda

In [ ]:
# Branch 1 (MambaIRv2 GPU Scan) — UIEB (200 epochs from scratch)
!cd /kaggle/working/exp1 && python3 train_kaggle.py \
    --dataset uieb \
    --scan_mode mambairv2 \
    --epochs 200 \
    --batch_size 8 \
    --lr 3e-4 \
    --img_size 256 \
    --amp \
    --seed 42 \
    --save_dir /kaggle/working/checkpoints/exp1_uieb

In [ ]:
# Branch 1 (MambaIRv2 GPU Scan) — LSUI (200 epochs from scratch)
!cd /kaggle/working/exp1 && python3 train_kaggle.py \
    --dataset lsui \
    --scan_mode mambairv2 \
    --epochs 200 \
    --batch_size 8 \
    --lr 3e-4 \
    --img_size 256 \
    --amp \
    --seed 42 \
    --save_dir /kaggle/working/checkpoints/exp1_lsui

In [ ]:
# Branch 2 (Physics-Guided Attenuation Scan + AT Cycle Loss) — UIEB (200 epochs from scratch)
!cd /kaggle/working/exp2 && python3 train_kaggle.py \
    --dataset uieb \
    --scan_mode physics_attenuation \
    --epochs 200 \
    --batch_size 8 \
    --lr 3e-4 \
    --img_size 256 \
    --amp \
    --lambda_phys 0.1 \
    --seed 42 \
    --save_dir /kaggle/working/checkpoints/exp2_uieb

In [ ]:
# Branch 2 (Physics-Guided Attenuation Scan + AT Cycle Loss) — LSUI (200 epochs from scratch)
!cd /kaggle/working/exp2 && python3 train_kaggle.py \
    --dataset lsui \
    --scan_mode physics_attenuation \
    --epochs 200 \
    --batch_size 8 \
    --lr 3e-4 \
    --img_size 256 \
    --amp \
    --lambda_phys 0.1 \
    --seed 42 \
    --save_dir /kaggle/working/checkpoints/exp2_lsui

In [ ]:
import json
from pathlib import Path

for run in ["exp1_uieb", "exp1_lsui", "exp2_uieb", "exp2_lsui"]:
    p = Path("/kaggle/working/checkpoints") / run / "eval_summary.json"
    if not p.is_file():
        print(f"{run}: NOT FOUND")
        continue
    d = json.loads(p.read_text())
    m = d["test_metrics"]
    hw = d["hardware_profile"]
    p256 = hw["256x256"]
    p720 = hw["1280x720"]
    print(
        f"{run:10s} | PSNR(RGB/Y)={m['psnr_rgb']:.2f}/{m['psnr_y']:.2f} dB | "
        f"SSIM(RGB/Y)={m['ssim_rgb']:.4f}/{m['ssim_y']:.4f} | "
        f"Params={hw['total_params_M']:.3f}M | GFLOPs={p256['gflops']} | "
        f"256={p256['fps']:.1f} FPS ({p256['latency_ms']:.1f} ms) | "
        f"720p={p720['fps']:.1f} FPS | VRAM={p256['peak_vram_mb']:.0f} MB | "
        f"JIT={hw['torch_jit_trace']['passed']}"
    )